# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [5]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [3]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [7]:
# TODO：创建 w1、w2、b，以及包含三个参数的 SGD 优化器
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y  = torch.tensor([-6., -1., -5., 6., -4., 4.])
                  
def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b

w1 = torch.tensor(0.0, requires_grad=True)   
w2 = torch.tensor(0.0, requires_grad=True)   
b  = torch.tensor(0.0, requires_grad=True)

optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

epoch = 2000
for epoch in range(1, 2001):
    lr = 0.1 if epoch <= 1000 else 0.001
    optimizer.param_groups[0]["lr"] = lr

    optimizer.zero_grad()
    y_pred = predict(x1, x2, w1, w2, b)      # 一次预测同时用 x1 和 x2
    loss = torch.abs(y - y_pred).mean()      # 先逐样本绝对误差，再取平均
    loss.backward()
    optimizer.step()

    if epoch in (1, 10, 100, 500, 1000, 1500, 2000):
        with torch.no_grad():
            current = torch.abs(y - predict(x1, x2, w1, w2, b)).mean().item()
        print(f"第{epoch:5d}轮  lr={lr:<6g} w1={w1.item(): .6f}  w2={w2.item(): .6f}  "
              f"b={b.item(): .6f}  平均绝对误差={current:.6f}")

# 用训练结束后的参数重新计算预测值和误差
with torch.no_grad():
    y_pred = predict(x1, x2, w1, w2, b)
    final_loss = torch.abs(y - y_pred).mean()

print("\n学到的参数：w1 =", w1.item(), " w2 =", w2.item(), " b =", b.item())
print("\n  x1     x2     目标 y     预测 y_hat     绝对误差")
for i in range(len(x1)):
    print(f"{x1[i].item():5.1f}  {x2[i].item():5.1f}  {y[i].item():8.3f}  "
          f"{y_pred[i].item():11.6f}  {abs(y[i].item() - y_pred[i].item()):10.6f}")
print("\n最终平均绝对误差：", final_loss.item())

第    1轮  lr=0.1    w1= 0.083333  w2=-0.100000  b=-0.033333  平均绝对误差=4.152778
第   10轮  lr=0.1    w1= 0.800000  w2=-1.000000  b=-0.300000  平均绝对误差=2.600000
第  100轮  lr=0.1    w1= 1.933333  w2=-3.016666  b= 0.950000  平均绝对误差=0.127778
第  500轮  lr=0.1    w1= 1.933333  w2=-3.016666  b= 0.950000  平均绝对误差=0.127778
第 1000轮  lr=0.1    w1= 1.933333  w2=-3.016666  b= 0.950000  平均绝对误差=0.127778
第 1500轮  lr=0.001  w1= 2.000834  w2=-2.999834  b= 1.000166  平均绝对误差=0.001416
第 2000轮  lr=0.001  w1= 2.000834  w2=-2.999834  b= 1.000166  平均绝对误差=0.001416

学到的参数：w1 = 2.000833511352539  w2 = -2.999833822250366  b = 1.0001659393310547

  x1     x2     目标 y     预测 y_hat     绝对误差
 -2.0    1.0    -6.000    -6.001335    0.001335
 -1.0    0.0    -1.000    -1.000668    0.000668
  0.0    2.0    -5.000    -4.999502    0.000498
  1.0   -1.0     6.000     6.000834    0.000834
  2.0    3.0    -4.000    -3.997668    0.002332
  3.0    1.0     4.000     4.002832    0.002832

最终平均绝对误差： 0.0014164447784423828


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

待填写。


In [ ]:
一次预测需要 2 个输入张量,x1、x2 的 shape 都是 (6,)，其中 6 是样本数
预测公式上周是 ŷ = w₁x + b，这周是 ŷ = w₁x₁ + w₂x₂ + b
参数列表：从 (w, b) 变成 (w1, w2, b)
损失计算：写法没变,变化的只是 y_pred 现在由两个输入共同算出
优化器步骤：没有变
平均损失显著下降并趋近,预测与目标一致,参数稳定,有理论对照
因为 0 只是起点，不是驻点,上周的那个函数里 0 恰好就是梯度为 0 的驻点，参数会永远停在原地，本周的损失是凸的分段线性函数